In [ ]:
### Imports

from pathlib import Path
import pandas as pd
import numpy as np
import tensorflow as tf
import keras
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

In [ ]:
data_dir = Path("../NeuralNetworks_project_work/archive/images")

counts = []
for artist_dir in data_dir.iterdir():
    if artist_dir.is_dir():
        n = len(list(artist_dir.glob("*")))
        counts.append((artist_dir.name, n))

df = pd.DataFrame(counts, columns=["artist", "image_count"])
df = df.sort_values("image_count", ascending=False)

print(df)
print("\nSmallest class:", df.image_count.min())
print("Largest class:", df.image_count.max())
print("Median class:", df.image_count.median())

In [ ]:
artist_counts = {
    "Vincent_van_Gogh": 877,
    "Edgar_Degas": 702,
    "Albrecht_Durer": 656,
    "Pablo_Picasso": 439,
    "Pierre-Auguste_Renoir": 336,
    "Albrecht_Durer": 328,
    "Paul_Gauguin": 311,
    "Francisco_Goya": 291,
    "Rembrandt": 262,
    "Alfred_Sisley": 259,
    "Titian": 255,
    "Marc_Chagall": 239,
    "Rene_Magritte": 194,
    "Amedeo_Modigliani": 193,
    "Paul_Klee": 188,
    "Henri_Matisse": 186,
    "Andy_Warhol": 181,
    "Mikhail_Vrubel": 171,
    "Sandro_Botticelli": 164,
    "Leonardo_da_Vinci": 143,
    "Peter_Paul_Rubens": 141,
    "Salvador_Dali": 139,
    "Hieronymus_Bosch": 137,
    "Pieter_Bruegel": 134,
    "Diego_Velazquez": 128,
    "Kazimir_Malevich": 126,
    "Frida_Kahlo": 120,
    "Giotto_di_Bondone": 119,
    "Gustav_Klimt": 117,
    "Raphael": 109,
    "Joan_Miro": 102,
    "Andrei_Rublev": 99,
    "Camille_Pissarro": 91,
    "Edouard_Manet": 90,
    "Vasiliy_Kandinskiy": 88,
    "El_Greco": 87,
    "Piet_Mondrian": 84,
    "Henri_de_Toulouse-Lautrec": 81,
    "Jan_van_Eyck": 81,
    "Claude_Monet": 73,
    "Diego_Rivera": 70,
    "Henri_Rousseau": 70,
    "Edvard_Munch": 67,
    "William_Turner": 66,
    "Gustave_Courbet": 59,
    "Caravaggio": 55,
    "Michelangelo": 49,
    "Paul_Cezanne": 47,
    "Georges_Seurat": 43,
    "Eugene_Delacroix": 31,
    "Jackson_Pollock": 24,
}

max_count = max(artist_counts.values())

soft_class_weights = {
    artist: np.sqrt(max_count / count)
    for artist, count in artist_counts.items()
}

soft_class_weights

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    "archive/images",
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "archive/images",
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)


train_ds = tf.keras.utils.image_dataset_from_directory(
    "archive/images",
    image_size=(224, 224),
    batch_size=32
)

class_names = train_ds.class_names
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
print(class_names)

In [ ]:
max_count = max(artist_counts.values())

class_weights = {
    i: float(np.sqrt(max_count / artist_counts[class_name]))
    for i, class_name in enumerate(class_names)
}

print(class_weights)

In [ ]:
num_classes = len(class_names)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomContrast(0.08),
])

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = tf.keras.Sequential([
    data_augmentation,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")
    ]
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights
)

In [ ]:
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

# PLOT
plt.figure(figsize=(20, 20))

plt.imshow(cm, interpolation='nearest')
plt.title("Artist Confusion Matrix")

plt.colorbar()

tick_marks = np.arange(len(class_names))

plt.xticks(
    tick_marks,
    class_names,
    rotation=90,
    fontsize=8
)

plt.yticks(
    tick_marks,
    class_names,
    fontsize=8
)

plt.xlabel("Predicted Artist")
plt.ylabel("True Artist")

plt.tight_layout()

plt.show()

In [ ]:
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

top_confusions = np.dstack(
    np.unravel_index(
        np.argsort(cm_no_diag.ravel())[::-1],
        cm_no_diag.shape
    )
)[0]

for true_idx, pred_idx in top_confusions[:20]:
    if cm_no_diag[true_idx, pred_idx] > 0:
        print(
            f"{class_names[true_idx]} predicted as {class_names[pred_idx]}: "
            f"{cm_no_diag[true_idx, pred_idx]} times"
        )